# Advanced RAG — Interview Deep Dive
### Every hard question answered with working code, step by step

These are the real questions that get asked in senior RAG interviews for legal AI.  
We're going to answer each one **by building it** — not by explaining it in theory.

---

### The Questions We're Answering:

| # | Question |
|---|---|
| 1 | Legal docs have images, tables, text — how do you retrieve them with **exact page + position**? |
| 2 | What **embedding models** do you use for multimodal legal RAG? |
| 3 | One document references another — how do you handle **cross-document links**? |
| 4 | What is **SPARQL** and where does it fit in a legal RAG system? |
| 5 | What are **VLMs** and which ones do you use? |
| 6 | How do you build **lossless** legal evaluation — zero hallucination, zero missed points? |
| 7 | Terms: **structure-aware chunking**, **graph algorithms** — what do these really mean? |

---

### What we already have:
- `eu_ai_act.pdf` parsed with fitz into blocks → chunks → `all_chunks.json`
- ChromaDB vector store with those chunks embedded
- Agentic RAG pipeline (CRAG + Self-RAG + multi-query)

### What we're building now:
- **Multimodal extraction** — pull images and tables out of the PDF with their exact bounding boxes
- **Structure-aware chunking** — understand what this actually means vs what we did before
- **Knowledge graph** — build Article → Annex → Recital cross-reference graph with rdflib
- **SPARQL queries** — query that graph like a database
- **VLM understanding** — what GPT-4V / LLaVA / Qwen-VL actually do with table images
- **Lossless evaluation** — coverage score, citation tracing, contradiction detection

Let's go.

---
# QUESTION 1
## Legal documents have images, tables, text — retrieve them with exact page + position

The interviewer is asking about **multimodal RAG**. The key insight is:  
a legal document isn't just text. A table on page 47 might define exactly which AI systems  
are high-risk. If you only chunk text and miss that table, your answer is wrong.

**The architecture the interviewer wants to hear:**
1. Parse the PDF at the block level — fitz already does this, but we need to identify *what kind* of block
2. Extract images as pixel data with their bounding boxes (page number + x,y,w,h)
3. Extract table regions — either via heuristics or a layout model (like `pdfplumber` or `TATR`)
4. Store each modality separately with position metadata
5. At retrieval time, return not just text but "here is the table on page 47, rows 3–7"

Let's first see what fitz actually gives us about images inside the EU AI Act PDF.

In [ ]:
import subprocess, sys

pkgs = [
    "pymupdf",          # fitz
    "pdfplumber",       # table extraction
    "rdflib",           # knowledge graph + SPARQL
    "networkx",         # graph algorithms
    "matplotlib",
    "Pillow",           # image handling
]
for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        capture_output=True, text=True
    )
print("All packages ready ✓")

In [ ]:
import fitz
import json
from pprint import pprint

PDF_PATH = "eu_ai_act.pdf"
doc = fitz.open(PDF_PATH)

print(f"Total pages: {len(doc)}")
print()

# Let's scan ALL pages and find pages that have embedded images
pages_with_images = []

for page_num in range(len(doc)):
    page = doc[page_num]
    images = page.get_images(full=True)
    if images:
        pages_with_images.append((page_num + 1, len(images), images))

print(f"Pages containing embedded images: {len(pages_with_images)}")
print()

# Let's look at the first few
for pg, count, imgs in pages_with_images[:5]:
    print(f"  Page {pg:3d} — {count} image(s)")
    for img in imgs:
        # get_images returns: (xref, smask, width, height, bpc, colorspace, alt_colorspace, name, filter, referencer)
        xref, smask, w, h, bpc, cs, alt_cs, name, filt, ref = img
        print(f"    xref={xref}, size={w}x{h}px, colorspace={cs}, filter={filt}")

In [ ]:
# Now let's go further — for each image, find its EXACT POSITION on the page
# This is what the interviewer means by "which page, which position"
# fitz has page.get_image_rects(xref) for this

print("Extracting image positions (bounding boxes)...")
print()

image_inventory = []  # this will be our multimodal chunk store

for page_num in range(len(doc)):
    page = doc[page_num]
    images = page.get_images(full=True)
    
    for img_index, img in enumerate(images):
        xref = img[0]
        width, height = img[2], img[3]
        
        # Get the position of this image on the page
        try:
            rects = page.get_image_rects(xref)
        except:
            rects = []
        
        for rect in rects:
            # rect is a fitz.Rect: (x0, y0, x1, y1) in points (72 pts = 1 inch)
            # Page is typically 595 x 842 points (A4)
            image_record = {
                "content_type": "image",
                "page_num": page_num + 1,
                "xref": xref,
                "pixel_width": width,
                "pixel_height": height,
                "bbox": {
                    "x0": round(rect.x0, 2),
                    "y0": round(rect.y0, 2),
                    "x1": round(rect.x1, 2),
                    "y1": round(rect.y1, 2),
                },
                "bbox_normalized": {
                    # Normalize to 0-1 range relative to page size
                    # Page width = 595, height = 842 (A4)
                    "x0": round(rect.x0 / 595, 4),
                    "y0": round(rect.y0 / 842, 4),
                    "x1": round(rect.x1 / 595, 4),
                    "y1": round(rect.y1 / 842, 4),
                },
                "source_ref": f"Image on p.{page_num + 1} at ({rect.x0:.0f},{rect.y0:.0f})",
                # We'll add caption text and embedding later
                "caption": "",
                "vlm_description": ""  # placeholder for VLM output
            }
            image_inventory.append(image_record)

print(f"Total image regions found: {len(image_inventory)}")
print()
if image_inventory:
    print("First image record:")
    pprint(image_inventory[0])
else:
    print("This PDF appears to have no embedded bitmap images.")
    print("The EU AI Act is a text-only document from EUR-Lex.")
    print("But the architecture still applies — let's simulate it with tables.")

In [ ]:
# The EU AI Act is mostly text, but it does have TABLES.
# Tables are harder to detect because fitz sees them as text blocks.
# pdfplumber has a table detection algorithm.

import pdfplumber

print("Scanning for tables with pdfplumber...")
print("(pdfplumber detects tables by looking for lines that form grid patterns)")
print()

table_inventory = []

with pdfplumber.open(PDF_PATH) as plumber_doc:
    for page_num, page in enumerate(plumber_doc.pages):
        tables = page.extract_tables()
        if tables:
            for table_idx, table in enumerate(tables):
                # Filter out empty tables or single-row headers
                non_empty_rows = [row for row in table if any(cell for cell in row if cell)]
                if len(non_empty_rows) < 2:
                    continue
                
                # Get the bounding box via page's table object
                # pdfplumber also gives us bbox when we use find_tables()
                table_record = {
                    "content_type": "table",
                    "page_num": page_num + 1,
                    "table_index": table_idx,
                    "rows": len(non_empty_rows),
                    "cols": max(len(row) for row in non_empty_rows),
                    "data": non_empty_rows,  # the actual cell content
                    # Convert table to a text representation for embedding
                    "text_repr": "\n".join(
                        " | ".join(cell or "" for cell in row)
                        for row in non_empty_rows
                    ),
                    "source_ref": f"Table {table_idx+1} on p.{page_num+1}",
                }
                table_inventory.append(table_record)

print(f"Total tables found: {len(table_inventory)}")
print()
for t in table_inventory[:3]:
    print(f"  Page {t['page_num']}, Table {t['table_index']+1}: {t['rows']} rows × {t['cols']} cols")
    print(f"  Source: {t['source_ref']}")
    print(f"  Preview:")
    for row in t['data'][:3]:
        print(f"    {row}")
    print()

In [ ]:
# pdfplumber's find_tables() also gives exact bounding boxes
# This is crucial for the interviewer's question: "which page, which position"

print("Getting exact bounding boxes for tables...")
print()

table_inventory_with_bbox = []

with pdfplumber.open(PDF_PATH) as plumber_doc:
    for page_num, page in enumerate(plumber_doc.pages):
        found = page.find_tables()
        for t_obj in found:
            bbox = t_obj.bbox  # (x0, top, x1, bottom) in PDF coordinate points
            table_data = t_obj.extract()
            non_empty = [r for r in table_data if any(c for c in r if c)]
            if len(non_empty) < 2:
                continue
            
            record = {
                "content_type": "table",
                "page_num": page_num + 1,
                "bbox": {
                    "x0": round(bbox[0], 2),
                    "y0": round(bbox[1], 2),
                    "x1": round(bbox[2], 2),
                    "y1": round(bbox[3], 2),
                },
                "page_width":  round(page.width, 2),
                "page_height": round(page.height, 2),
                "rows": len(non_empty),
                "cols": max(len(r) for r in non_empty),
                "data": non_empty,
                "text_repr": "\n".join(" | ".join(c or "" for c in r) for r in non_empty),
                "source_ref": f"Table on p.{page_num+1} at ({bbox[0]:.0f},{bbox[1]:.0f})",
            }
            table_inventory_with_bbox.append(record)

print(f"Tables with bounding boxes: {len(table_inventory_with_bbox)}")
print()
for t in table_inventory_with_bbox[:2]:
    print(f"  Page {t['page_num']} | bbox={t['bbox']}")
    print(f"  Size: {t['rows']} rows × {t['cols']} cols | Page dims: {t['page_width']}x{t['page_height']}")
    print(f"  Text repr preview:")
    print("  " + "\n  ".join(t['text_repr'][:300].split("\n")[:4]))
    print()

In [ ]:
# Now let's define the UNIFIED multimodal chunk schema
# This is what the interviewer wants — a single schema that covers all modalities
# with position metadata that can be returned to the user

from dataclasses import dataclass, asdict
from typing import Optional, Literal

@dataclass
class MultimodalChunk:
    """A chunk that can be text, table, or image — with exact provenance."""
    
    # --- Identity ---
    chunk_id: str
    content_type: Literal["text", "table", "image"]   # modality
    
    # --- What to embed ---
    embed_text: str     # text/table: the text itself; image: VLM-generated caption
    
    # --- Exact provenance (the answer to the interview question) ---
    document_name: str  # which PDF
    page_num: int       # which page
    bbox: Optional[dict] = None  # {x0, y0, x1, y1} in PDF points
    
    # --- Structural context ---
    article: Optional[str] = None
    chapter: Optional[str] = None
    section: Optional[str] = None
    
    # --- Modality-specific ---
    table_data: Optional[list] = None   # raw rows for tables
    image_xref: Optional[int] = None    # fitz xref for images
    vlm_description: Optional[str] = None  # GPT-4V / LLaVA output
    
    # --- Cross-references ---
    references: Optional[list] = None   # ["Article 6", "Annex III"]

# Let's create one example of each type from what we've found
import hashlib

def make_id(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()[:12]

# Example text chunk (from our existing all_chunks.json)
text_chunk = MultimodalChunk(
    chunk_id=make_id("Article 6 p.55"),
    content_type="text",
    embed_text="High-risk AI systems referred to in Annex III shall comply with the requirements...",
    document_name="eu_ai_act.pdf",
    page_num=55,
    bbox={"x0": 105.8, "y0": 120.0, "x1": 489.4, "y1": 240.0},
    article="Article 6",
    chapter="CHAPTER III",
    references=["Annex III", "Article 7"]
)

# Example table chunk
if table_inventory_with_bbox:
    t = table_inventory_with_bbox[0]
    table_chunk = MultimodalChunk(
        chunk_id=make_id(t['source_ref']),
        content_type="table",
        embed_text=t['text_repr'],
        document_name="eu_ai_act.pdf",
        page_num=t['page_num'],
        bbox=t['bbox'],
        table_data=t['data']
    )
    print("Text chunk:")
    print(f"  type={text_chunk.content_type}, page={text_chunk.page_num}, bbox={text_chunk.bbox}")
    print()
    print("Table chunk:")
    print(f"  type={table_chunk.content_type}, page={table_chunk.page_num}, bbox={table_chunk.bbox}")
    print(f"  embed_text[:100]: {table_chunk.embed_text[:100]}")
else:
    print("Text chunk example:")
    pprint(asdict(text_chunk))

print()
print("KEY INSIGHT for interview:")
print("  Every chunk — text, table, image — carries page_num + bbox.")
print("  When the RAG returns an answer, it can say:")
print("  'This comes from Table on page 47 at position (105, 320, 489, 450)'")
print("  Which means you can highlight it in the original PDF for the user.")

---
# QUESTION 2
## What embedding models do you use for multimodal legal RAG?

This is where most candidates say "OpenAI embeddings" and stop. The interviewer wants depth.

**The real answer has three layers:**

### Layer 1 — Text embeddings (for text + table chunks)

| Model | Why | When to use |
|---|---|---|
| `BAAI/bge-m3` | Best open-source legal embedding. 8192 token context. Dense + sparse + colbert in ONE model | Production legal RAG |
| `intfloat/e5-large-v2` | Very good, lighter than BGE-M3 | Resource-constrained |
| `nlpaueb/legal-bert-base-uncased` | Fine-tuned on legal corpora specifically | If your domain is purely legal |
| `text-embedding-3-large` (OpenAI) | High quality, no infra | API-based systems |

### Layer 2 — Image embeddings (for image chunks)

| Model | Why |
|---|---|
| `CLIP` (OpenAI) | Embeds images and text in the SAME vector space — you can retrieve images with text queries |
| `jina-clip-v1` | Better than CLIP for documents |

### Layer 3 — Cross-modal late fusion

Text query → search text index (BGE-M3) + image index (CLIP) → RRF merge → unified results

Let's actually load BGE-M3 and see what it produces.

In [ ]:
# BGE-M3: The flagship legal embedding model
# It produces THREE types of vectors from the SAME model:
#   1. dense vector (for cosine similarity search)
#   2. sparse vector (BM25-like lexical scores)
#   3. colbert multi-vector (per-token — most expressive but expensive)

# Let's install the FlagEmbedding library which gives us all three in one call

import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "FlagEmbedding", "-q", "--break-system-packages"],
    capture_output=True, text=True
)
print(result.stdout[-200:] if result.stdout else "installed")

# Also install sentence-transformers for backup
result2 = subprocess.run(
    [sys.executable, "-m", "pip", "install", "sentence-transformers", "-q", "--break-system-packages"],
    capture_output=True, text=True
)
print("sentence-transformers ready ✓")

In [ ]:
# Let's use sentence-transformers with BGE to understand the embedding space
# We'll compare three models on the same legal query to see the difference

from sentence_transformers import SentenceTransformer
import numpy as np

# Load a smaller but still legal-aware model for demo
# (BGE-M3 is 568MB, we'll use BGE-small for speed)
print("Loading BAAI/bge-small-en-v1.5 (lightweight BGE for demo)...")
model_bge = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"  Embedding dimension: {model_bge.get_sentence_embedding_dimension()}")
print()

# Test sentences — legal query vs. matching chunks vs. irrelevant
query = "What obligations does a provider of high-risk AI have?"

candidates = [
    # -- highly relevant --
    "Providers of high-risk AI systems shall ensure compliance with requirements set out in Chapter III.",
    "The provider shall establish a quality management system and keep technical documentation.",
    # -- medium relevant --
    "Deployers of high-risk AI systems shall monitor the operation of the AI system.",
    # -- low relevant --
    "This Regulation enters into force twenty days after its publication in the Official Journal.",
    "The Commission shall review the application of the Regulation every four years.",
]

# BGE models need a special prefix for queries (not for documents)
query_with_prefix = f"Represent this sentence for searching relevant passages: {query}"

q_emb = model_bge.encode(query_with_prefix, normalize_embeddings=True)
c_embs = model_bge.encode(candidates, normalize_embeddings=True)

# Cosine similarity (since normalized, dot product = cosine)
scores = c_embs @ q_emb

print(f"Query: '{query}'")
print()
print("BGE similarity scores:")
for score, text in sorted(zip(scores, candidates), reverse=True):
    bar = "█" * int(score * 40)
    print(f"  {score:.4f} {bar}")
    print(f"           {text[:80]}")
    print()

print("OBSERVATION: BGE correctly ranks 'provider obligations' above 'entry into force'.")
print("But notice it also scores the DEPLOYER sentence fairly high — because semantically")
print("it's about roles and obligations too. This is where legal-specific fine-tuning matters.")

In [ ]:
# Now let's understand what CLIP does for images
# CLIP embeds images AND text in the SAME space
# So a text query can retrieve images directly

# For this demo, we'll show the architecture without loading the full model
# (CLIP is 400MB and needs torch CUDA for speed)

print("=" * 60)
print("CLIP Architecture for Multimodal Legal RAG")
print("=" * 60)
print()
print("How CLIP works:")
print("  Text: 'table showing prohibited AI systems'")
print("        ↓ text encoder")
print("        [0.12, -0.45, 0.78, ...] (512-dim)")
print()
print("  Image: (pixel data of table)")
print("         ↓ vision encoder (ViT-B/32)")
print("         [0.09, -0.41, 0.81, ...] (512-dim)")
print()
print("  Cosine similarity: 0.89 → HIGH MATCH")
print("  → System retrieves the image chunk")
print()
print("In practice for legal docs:")
print("  1. Extract all images from PDF with fitz")
print("  2. Pass each through CLIP vision encoder → image_embedding")
print("  3. ALSO run a VLM (GPT-4V/LLaVA) to generate a text caption")
print("  4. Embed the caption with BGE-M3 → text_embedding")
print("  5. Store BOTH embeddings for the same image chunk")
print("  6. At retrieval: search both indexes, RRF merge")
print()
print("This gives you TWO retrieval paths for the same image:")
print("  Path A: User query matches CLIP vision embedding directly")
print("  Path B: User query matches the VLM-generated caption")
print()
print("Interview answer: We use BGE-M3 for text, CLIP for images,")
print("and late fusion (RRF) to merge results from both indexes.")

In [ ]:
# Let's also talk about BGE-M3's HYBRID capability
# This is the key differentiator for legal RAG

print("BGE-M3 Hybrid: Dense + Sparse + ColBERT from ONE model")
print("=" * 55)
print()
print("Problem with pure dense search on legal text:")
print("  Query: 'Article 5(1)(a)'")
print("  Dense search: might find semantically related paragraphs")
print("                but miss the EXACT citation Article 5(1)(a)")
print()
print("Problem with pure BM25 on legal queries:")
print("  Query: 'What can't an AI system do in public spaces?'")
print("  BM25: misses 'prohibited AI systems real-time biometric surveillance'")
print("        because the user didn't use those exact words")
print()
print("BGE-M3 solution:")
print("  Dense vector:  captures semantic meaning")
print("  Sparse vector: captures exact term overlap (like BM25)")
print("  ColBERT:       per-token late interaction — most precise")
print()
print("Final hybrid score = w1*dense + w2*sparse + w3*colbert")
print("For legal RAG: w1=0.4, w2=0.2, w3=0.4 (prefer precision)")
print()
print("This is why BGE-M3 is the right choice for legal documents.")

---
# QUESTION 3 + 4
## Cross-document references + SPARQL

This is the most sophisticated question. The interviewer is asking about **knowledge graphs**.

In the EU AI Act:
- Article 6 → references Annex III
- Annex III → references specific AI system categories
- Recital (57) → explains Article 16
- Article 112 → references Articles 5, 7, 46, 74

And across documents:
- EU AI Act → references GDPR (Regulation (EU) 2016/679)
- EU AI Act → references EU Machinery Directive
- EU AI Act → references EU Medical Device Regulation

**The architecture:**
1. Build a Knowledge Graph where nodes = documents/articles/sections, edges = references
2. Use RDF (Resource Description Framework) to represent this graph
3. Query it with SPARQL — a graph query language
4. During retrieval: when you find a chunk, also fetch its 1-hop neighbors in the graph

This is called **Graph RAG** and it's what separates senior candidates.

In [ ]:
# rdflib lets us build a knowledge graph using RDF triples
# RDF triple format: (subject, predicate, object)
# Example: (Article_6, references, Annex_III)
#          (Article_6, belongsTo, Chapter_III)
#          (Recital_57, explains, Article_16)

import rdflib
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS
from rdflib.namespace import XSD

print(f"rdflib version: {rdflib.__version__}")
print()

# Define our namespaces
# A namespace is just a URI prefix for our concepts
EUAI = Namespace("http://euaiact.eu/ontology#")
DOC  = Namespace("http://euaiact.eu/document/")

# Create the graph
g = Graph()
g.bind("euai", EUAI)
g.bind("doc",  DOC)

# Define our node types
# (These are like classes in OOP but for knowledge graphs)
TYPES = {
    "Article":   EUAI.Article,
    "Annex":     EUAI.Annex,
    "Recital":   EUAI.Recital,
    "Chapter":   EUAI.Chapter,
    "ExternalDoc": EUAI.ExternalDocument,
}

# Define our relationship types (predicates)
PREDS = {
    "references":   EUAI.references,         # explicit cross-reference
    "explains":     EUAI.explains,            # recital → article
    "belongsTo":    EUAI.belongsTo,           # article → chapter
    "amends":       EUAI.amends,              # article → external regulation
    "implements":   EUAI.implements,          # article → directive
    "has_title":    EUAI.hasTitle,
    "has_page":     EUAI.hasPage,
}

print("Namespaces defined:")
print(f"  EUAI: {EUAI}")
print(f"  DOC:  {DOC}")
print()
print("Node types:", list(TYPES.keys()))
print("Predicates:", list(PREDS.keys()))

In [ ]:
# Now let's build the graph from our all_chunks.json
# We'll extract the cross-references that we stored in metadata

import json, re

with open("./chunker_output/all_chunks.json") as f:
    all_chunks = json.load(f)

print(f"Loaded {len(all_chunks)} chunks")
print()

# Helper: convert article name to a URI-safe node ID
def to_node_id(name: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_]', '_', name.strip())

def add_node(g, node_id: str, node_type: str, title: str = "", page: int = 0):
    node = DOC[node_id]
    g.add((node, RDF.type, TYPES.get(node_type, EUAI.Node)))
    if title:
        g.add((node, PREDS["has_title"], Literal(title)))
    if page:
        g.add((node, PREDS["has_page"], Literal(page, datatype=XSD.integer)))
    return node

# Build the graph: iterate over all chunks
articles_added = set()
chapters_added = set()
recitals_added = set()
edge_count = 0

for chunk in all_chunks:
    meta = chunk["metadata"]
    ctype = meta.get("content_type", "")
    
    if ctype == "article":
        article_info = meta.get("article", {})
        art_name = article_info.get("article_name", "")
        if not art_name or art_name in articles_added:
            continue
        articles_added.add(art_name)
        
        art_id = to_node_id(art_name)
        art_node = add_node(g, art_id, "Article", 
                            title=article_info.get("article_title", ""),
                            page=article_info.get("page_num", 0))
        
        # Add chapter membership
        chap_name = article_info.get("chapter_title", "")
        if chap_name and chap_name not in chapters_added:
            chapters_added.add(chap_name)
            add_node(g, to_node_id(chap_name), "Chapter", title=chap_name)
        
        if chap_name:
            g.add((art_node, PREDS["belongsTo"], DOC[to_node_id(chap_name)]))
            edge_count += 1
        
        # Add all references THIS article makes
        refs = article_info.get("referenced_articles", []) + article_info.get("referenced_annexes", [])
        for ref in refs:
            ref_id = to_node_id(ref)
            ref_type = "Annex" if "Annex" in ref else "Article"
            # Add the referenced node if not seen yet
            add_node(g, ref_id, ref_type, title=ref)
            # Add the edge
            g.add((art_node, PREDS["references"], DOC[ref_id]))
            edge_count += 1
    
    elif ctype == "recital":
        recital_info = meta.get("recital", {})
        rec_num = recital_info.get("recital_num", 0)
        if rec_num in recitals_added:
            continue
        recitals_added.add(rec_num)
        
        rec_id = f"Recital_{rec_num}"
        rec_node = add_node(g, rec_id, "Recital",
                            title=f"Recital ({rec_num})",
                            page=recital_info.get("page_num", 0))
        
        # Recitals explain articles
        for ref_art in recital_info.get("referenced_articles", []):
            ref_id = to_node_id(ref_art)
            add_node(g, ref_id, "Article", title=ref_art)
            g.add((rec_node, PREDS["explains"], DOC[ref_id]))
            edge_count += 1

# Add external documents (from our prior knowledge of the EU AI Act)
external_docs = [
    ("GDPR",               "Regulation (EU) 2016/679"),
    ("MachDir",            "EU Machinery Directive 2006/42/EC"),
    ("MedDevice",          "EU Medical Device Regulation 2017/745"),
    ("NIS2",               "Directive (EU) 2022/2555 on Cybersecurity"),
    ("ProductSafety",      "Regulation (EU) 2023/988 on Product Safety"),
]
for doc_id, doc_name in external_docs:
    ext_node = add_node(g, doc_id, "ExternalDoc", title=doc_name)

# Article 5 references GDPR (known from reading the Act)
g.add((DOC["Article_5"], PREDS["references"], DOC["GDPR"]))
g.add((DOC["Article_10"], PREDS["references"], DOC["GDPR"]))
g.add((DOC["Article_9"],  PREDS["references"], DOC["MachDir"]))
edge_count += 3

print(f"Graph built!")
print(f"  Articles added:   {len(articles_added)}")
print(f"  Recitals added:   {len(recitals_added)}")
print(f"  Chapters added:   {len(chapters_added)}")
print(f"  External docs:    {len(external_docs)}")
print(f"  Total triples:    {len(g)}")
print(f"  Total edges:      {edge_count}")

In [ ]:
# Now — SPARQL.
# SPARQL is to knowledge graphs what SQL is to relational databases.
# You write graph pattern queries instead of table joins.

# SPARQL query 1: What does Article 6 reference?
q1 = """
PREFIX doc:  <http://euaiact.eu/document/>
PREFIX euai: <http://euaiact.eu/ontology#>

SELECT ?target ?title
WHERE {
    doc:Article_6  euai:references  ?target .
    OPTIONAL { ?target euai:hasTitle ?title . }
}
"""

print("SPARQL Query 1: What does Article 6 reference?")
print("-" * 50)
results = g.query(q1)
for row in results:
    target = str(row.target).split("/")[-1].replace("_", " ")
    title  = str(row.title) if row.title else ""
    print(f"  → {target}  {title}")
if not list(g.query(q1)):
    print("  (No results — Article_6 may not have explicit refs in our chunk metadata)")

In [ ]:
# SPARQL query 2: Which articles reference Annex III?
# This is the REVERSE lookup — crucial for legal RAG
# ("all articles that mention Annex III" → context expansion)

q2 = """
PREFIX doc:  <http://euaiact.eu/document/>
PREFIX euai: <http://euaiact.eu/ontology#>

SELECT ?article ?title
WHERE {
    ?article  euai:references  doc:Annex_III .
    OPTIONAL { ?article euai:hasTitle ?title . }
}
ORDER BY ?article
"""

print("SPARQL Query 2: Which articles reference Annex III?")
print("-" * 50)
results = list(g.query(q2))
for row in results:
    art = str(row.article).split("/")[-1].replace("_", " ")
    title = str(row.title) if row.title else ""
    print(f"  {art}: {title}")
print(f"\nTotal: {len(results)} articles reference Annex III")

In [ ]:
# SPARQL query 3: 2-hop path query
# "What recitals explain articles that reference Annex III?"
# This is graph traversal — exactly what the interviewer means by 'graph algorithms'

q3 = """
PREFIX doc:  <http://euaiact.eu/document/>
PREFIX euai: <http://euaiact.eu/ontology#>

SELECT ?recital ?article ?rec_title ?art_title
WHERE {
    ?article   euai:references  doc:Annex_III .
    ?recital   euai:explains    ?article .
    OPTIONAL { ?recital euai:hasTitle ?rec_title . }
    OPTIONAL { ?article euai:hasTitle ?art_title . }
}
ORDER BY ?recital
LIMIT 10
"""

print("SPARQL Query 3: 2-hop — Recitals that explain articles referencing Annex III")
print("(This is what cross-document graph traversal looks like)")
print("-" * 60)
results = list(g.query(q3))
for row in results:
    rec = str(row.recital).split("/")[-1].replace("_", " ")
    art = str(row.article).split("/")[-1].replace("_", " ")
    print(f"  {rec} → explains → {art} → references → Annex III")

if not results:
    print("  No 2-hop paths found — this means our metadata needs richer recital→article links")
    print("  But the architecture is correct: we'd find paths like:")
    print("  Recital(57) → explains → Article(16) → references → Annex(III)")

print()
print("HOW THIS HELPS RAG:")
print("  User asks: 'What AI systems are listed in Annex III?'")
print("  Step 1: Dense search finds Annex III chunks directly")
print("  Step 2: SPARQL finds all articles that reference Annex III")
print("  Step 3: Those articles' chunks are added to context")
print("  Step 4: SPARQL finds recitals explaining those articles")
print("  Step 5: Those recitals are also added")
print("  RESULT: Massively richer context, zero manual lookup")

In [ ]:
# Let's build the GRAPH RAG expansion function
# This is the function that gets called after normal retrieval
# to expand context using the knowledge graph

import re

def extract_article_refs(text: str) -> list:
    """Extract article/annex references from chunk text."""
    patterns = [
        r'Article\s+(\d+[a-z]?)',
        r'Articles\s+(\d+[a-z]?)(?:\s*,\s*(\d+[a-z]?))*',
        r'Annex\s+(I{1,3}V?|IV|[IVX]+)',
    ]
    found = []
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            found.append(match.group(0).strip())
    return list(set(found))

def graph_expand(retrieved_chunks: list, g: Graph, max_hops: int = 1) -> list:
    """
    Given a list of retrieved chunks, use the knowledge graph
    to find related chunks (1-hop or 2-hop neighbors).
    
    Returns: list of additional chunk identifiers to fetch
    """
    expansion_refs = set()
    
    for chunk in retrieved_chunks:
        text = chunk.get("page_content", "")
        meta = chunk.get("metadata", {})
        
        # Extract article references from the chunk text
        refs_in_text = extract_article_refs(text)
        
        # Also get refs from metadata
        article_info = meta.get("article", {})
        refs_in_meta = (
            article_info.get("referenced_articles", []) +
            article_info.get("referenced_annexes", [])
        )
        
        all_refs = set(refs_in_text + refs_in_meta)
        
        for ref in all_refs:
            ref_node_id = to_node_id(ref)
            
            # SPARQL: find what this referenced node ALSO references (1 more hop)
            if max_hops >= 1:
                q_expand = f"""
                PREFIX doc:  <http://euaiact.eu/document/>
                PREFIX euai: <http://euaiact.eu/ontology#>
                SELECT ?neighbor
                WHERE {{
                    doc:{ref_node_id}  euai:references  ?neighbor .
                }}
                """
                for row in g.query(q_expand):
                    neighbor_id = str(row.neighbor).split("/")[-1].replace("_", " ")
                    expansion_refs.add(neighbor_id)
            
            expansion_refs.add(ref)
    
    return list(expansion_refs)

# Test it with a sample chunk
sample_chunk = {
    "page_content": "Providers of high-risk AI systems as referred to in Article 6 and listed in Annex III shall comply with requirements of Article 9.",
    "metadata": {
        "article": {
            "article_name": "Article 16",
            "referenced_articles": ["Article 6", "Article 9", "Article 17"],
            "referenced_annexes": ["Annex III"]
        }
    }
}

expanded = graph_expand([sample_chunk], g, max_hops=1)
print("Graph expansion for a chunk mentioning Article 6, Annex III, Article 9:")
print(f"  Found {len(expanded)} related nodes to fetch:")
for ref in sorted(expanded):
    print(f"    → {ref}")

In [ ]:
# Save the graph as Turtle format (standard RDF serialization)
# This can be loaded into any triple store (Fuseki, GraphDB, Stardog)

g.serialize(destination="eu_ai_act_knowledge_graph.ttl", format="turtle")
print("Graph saved as eu_ai_act_knowledge_graph.ttl")
print()

# Show a snippet of the turtle format
with open("eu_ai_act_knowledge_graph.ttl") as f:
    content = f.read()

print("Turtle format sample (first 800 chars):")
print("-" * 50)
print(content[:800])
print("-" * 50)
print()
print(f"Total file size: {len(content)} bytes")
print()
print("INTERVIEW ANSWER for Q3 + Q4:")
print("  'We model cross-document references as an RDF knowledge graph.")
print("   Each article, recital, annex, and external regulation is a node.")
print("   References become typed edges (references, explains, amends, implements).")
print("   We query this with SPARQL for graph traversal during retrieval.")
print("   When a retrieved chunk mentions Article 6, SPARQL finds all chunks")
print("   related to Article 6 via 1- or 2-hop paths, expanding our context.")
print("   This is how we handle references to other documents like GDPR.")
print("   We add GDPR as external nodes in the graph and index GDPR chunks")
print("   in a separate collection — SPARQL tells us which ones to fetch.'")

---
# QUESTION 5
## What are VLMs and which ones do you use?

**VLM = Vision Language Model**

A VLM can look at an image and answer questions about it in natural language.  
In legal RAG, this solves the problem of tables and figures that are images.

### The models the interviewer wants to hear:

| Model | By | Best for | Cost |
|---|---|---|---|
| **GPT-4o** | OpenAI | Highest accuracy on complex tables | API |
| **Claude 3.5 Sonnet** | Anthropic | Legal reasoning + image reading | API |
| **Qwen2-VL** | Alibaba | Best open-source, strong on tables/charts | Self-host |
| **LLaVA-1.6** | LMSYS | Lightweight open-source VLM | Self-host |
| **InternVL2** | OpenGVLab | Excellent document understanding | Self-host |
| **ColPali** | HuggingFace | Embeds PDF pages as images directly | Self-host |

### ColPali deserves special mention
ColPali skips the whole extract-then-embed pipeline.  
It renders each PDF PAGE as an image, then embeds the entire page using a PaliGemma vision model.  
At retrieval, the user query is embedded and matched against page images directly.
**No parsing, no chunking, no layout detection.** Retrieval precision on document benchmarks beats everything else.

In [ ]:
# Let's build the VLM pipeline for table understanding
# We'll use fitz to extract table regions as images,
# then show what we'd send to a VLM

import fitz
from PIL import Image
import io, base64
import os

doc = fitz.open(PDF_PATH)

def extract_region_as_image(doc, page_num: int, bbox: dict, dpi: int = 150) -> Image.Image:
    """
    Extract a rectangular region from a PDF page as a PIL Image.
    bbox: {x0, y0, x1, y1} in PDF points (72 pts per inch)
    dpi:  resolution for rasterization
    """
    page = doc[page_num - 1]
    # Create a clip rectangle
    clip = fitz.Rect(bbox["x0"], bbox["y0"], bbox["x1"], bbox["y1"])
    
    # Render at higher resolution
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pixmap = page.get_pixmap(matrix=mat, clip=clip)
    
    # Convert to PIL Image
    img_bytes = pixmap.tobytes("png")
    img = Image.open(io.BytesIO(img_bytes))
    return img

def image_to_base64(img: Image.Image) -> str:
    """Convert PIL Image to base64 string for API calls."""
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode()

def render_full_page(doc, page_num: int, dpi: int = 120) -> Image.Image:
    """Render an entire page as an image (ColPali-style)."""
    page = doc[page_num - 1]
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pixmap = page.get_pixmap(matrix=mat)
    img_bytes = pixmap.tobytes("png")
    return Image.open(io.BytesIO(img_bytes))

# Extract page 1 as a demo
page_1_img = render_full_page(doc, page_num=1, dpi=100)
print(f"Full page 1 rendered: {page_1_img.size} pixels")

# If we have tables, extract the first one
if table_inventory_with_bbox:
    t = table_inventory_with_bbox[0]
    table_img = extract_region_as_image(doc, t['page_num'], t['bbox'], dpi=150)
    print(f"Table region extracted: {table_img.size} pixels")
    print(f"Table is on page {t['page_num']} at bbox {t['bbox']}")
    # Save for inspection
    table_img.save("sample_table_region.png")
    print("Saved as sample_table_region.png")
else:
    # Extract a region from page 1 as demo
    region_img = extract_region_as_image(doc, 1, {"x0": 80, "y0": 80, "x1": 520, "y1": 300}, dpi=150)
    region_img.save("sample_region.png")
    print(f"Sample region extracted: {region_img.size} pixels, saved as sample_region.png")

print()
print("This image would be sent to a VLM with a prompt like:")
print('  "Please extract all information from this table."')
print('  "List each row as: | col1 | col2 | col3 |"')
print('  "Also note any footnotes or special markings."')

In [ ]:
# The VLM pipeline for legal documents:
# 1. Extract region as image
# 2. Send to VLM with structured prompt
# 3. Get back structured text
# 4. Store as text_repr in the chunk for embedding

# Here's what the actual API call looks like (Claude's vision API)
# We'll show the structure without running it to avoid API costs

def build_vlm_request_claude(img: Image.Image, page_num: int, context: str = "") -> dict:
    """
    Build the request payload for Claude's vision API.
    Same pattern works for GPT-4o with slightly different format.
    """
    b64 = image_to_base64(img)
    
    system_prompt = """You are a legal document analyst. Extract information from the 
provided image with complete accuracy. Do not infer or add anything not visible."""
    
    user_prompt = f"""This image is from page {page_num} of the EU AI Act.
{f'Legal context: {context}' if context else ''}

Please:
1. If this is a TABLE: Extract all cells. Format as markdown table. Note headers clearly.
2. If this is a FIGURE/DIAGRAM: Describe what it shows, what entities are depicted, any labels.
3. If this is a CHART: Describe the axes, data series, key values.
4. Note the page number and any surrounding text that gives context.
5. Identify any cross-references (Article X, Annex Y, Recital Z) visible in or near the image.

Be exhaustive. A legal analyst will use your output to answer compliance questions."""

    return {
        "model": "claude-opus-4-5",
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": [{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": b64
                    }
                },
                {"type": "text", "text": user_prompt}
            ]
        }]
    }

# Show the structure (not running to save API cost)
sample_img = render_full_page(doc, 1, dpi=72)  # low res for demo
request_payload = build_vlm_request_claude(sample_img, page_num=1)

print("VLM Request Structure:")
print(f"  model: {request_payload['model']}")
print(f"  messages[0].content[0]: image/png ({len(request_payload['messages'][0]['content'][0]['source']['data'])} base64 chars)")
print(f"  messages[0].content[1]: text prompt ({len(request_payload['messages'][0]['content'][1]['text'])} chars)")
print()
print("User prompt sent to VLM:")
print("-" * 40)
print(request_payload['messages'][0]['content'][1]['text'])

In [ ]:
# ColPali: the paradigm shift for document RAG
# Instead of: parse → chunk → embed text
# ColPali does: render page → embed page image

print("ColPali Architecture")
print("=" * 55)
print()
print("Traditional RAG pipeline:")
print("  PDF → [fitz parse] → text blocks → [structure parsing] →")
print("  chunks → [BGE-M3 embed] → vector store")
print()
print("  Problems:")
print("    - Tables become garbled text")
print("    - Images are lost")
print("    - Layout information is lost")
print("    - Multi-column text gets merged")
print()
print("ColPali pipeline:")
print("  PDF → [render each page at ~300dpi] → page images →")
print("  [PaliGemma vision encoder] → page embeddings → vector store")
print()
print("  Advantages:")
print("    - Tables preserved exactly as they look")
print("    - Images embedded directly")
print("    - Layout preserved")
print("    - No parsing errors")
print()

# Render all pages as images (this is what ColPali indexes)
print("Rendering all pages as images...")
page_images = []
sample_pages = min(5, len(doc))  # just first 5 for demo

for pg_num in range(1, sample_pages + 1):
    img = render_full_page(doc, pg_num, dpi=100)
    page_images.append({
        "page_num": pg_num,
        "image": img,
        "size": img.size,
        "base64": image_to_base64(img)
    })

print(f"Rendered {len(page_images)} pages")
for p in page_images:
    print(f"  Page {p['page_num']}: {p['size']} pixels, {len(p['base64'])} base64 bytes")
print()
print("In ColPali: each of these pages gets embedded by PaliGemma → 128 vectors of 128-dim each.")
print("At query time: query is embedded similarly → max-sim matching across all page vectors.")
print("Result: page-level retrieval that understands tables AND text AND images together.")

---
# QUESTION 6 + 7
## Lossless Legal Evaluation + Terminology Deep Dive

This is the hardest question. The interviewer is probing whether you understand  
that legal RAG has **zero tolerance for hallucination or missing points**.

### The terminology the interviewer wants to hear:

**Structure-Aware Chunking:**  
Chunking that respects the document's logical structure — articles, paragraphs, sub-clauses —  
rather than splitting on character count. We already do this (that's the whole point of Chunking_v2).  
The key is: never split in the middle of a legal obligation. A provision is the atomic unit.

**Graph Algorithms in RAG:**
- PageRank on the citation graph → articles referenced by many others are more important
- BFS/DFS for cross-reference expansion (already showed this with SPARQL)
- Community detection → cluster articles into logical topics

**For lossless evaluation:**
- Citation coverage: every claim in the answer must trace to a source chunk
- Contradiction detection: answer must not contradict any retrieved chunk
- Completeness scoring: did we retrieve ALL relevant chunks or just some?
- Legal-specific: obligation/prohibition/permission triple extraction

In [ ]:
# STRUCTURE-AWARE CHUNKING — what it really means
# vs what we did before

import json

with open("./chunker_output/all_chunks.json") as f:
    chunks = json.load(f)

print("Structure-Aware Chunking vs Naive Chunking")
print("=" * 55)
print()

# Find a multi-chunk article to illustrate
art16_chunks = [c for c in chunks if 
                c['metadata'].get('content_type') == 'article' and
                c['metadata'].get('article', {}).get('article_name', '') == 'Article 16']

print(f"Article 16 has {len(art16_chunks)} chunk(s)")
if art16_chunks:
    for i, ch in enumerate(art16_chunks[:2]):
        text = ch['page_content']
        print(f"  Chunk {i+1}: {len(text)} chars")
        print(f"  Starts with: {text[:100]}...")
        print()

print("What NAIVE chunking would do (fixed 512 chars):")
naive_example = """
Article 16 — Obligations of providers of high-risk AI systems

Providers of high-risk AI systems shall:
(a) ensure that their high-risk AI systems are compliant with the requirements set out in Section 2;
(b) have a quality management system in place pursuant to Article 17;
(c) draw up the technical documentation of the high-risk AI system in accordance with Article 11
and Annex IV;
(d) when applicable, follow the relevant conformity assessment procedure in accordance with Article 43;
(e) register the high-risk AI system in the EU database;
(f) once it has been placed on the market or put into service, take the necessary corrective actions
and provide information as required pursuant to Article 20;
(g) upon request by a national competent authority, demonstrate the conformity of the high-risk AI system.
"""

# Naive split at 200 chars (to illustrate the problem)
CHUNK_SIZE = 200
naive_chunks = [naive_example[i:i+CHUNK_SIZE] for i in range(0, len(naive_example), CHUNK_SIZE)]

print()
print("Naive split result (200 char windows):")
for i, nc in enumerate(naive_chunks):
    print(f"  Chunk {i+1}: '{nc.strip()[:80]}...'")
    if 'Article 20' in nc and ';' not in nc:
        print(f"    ← NOTE: This chunk cuts mid-obligation!")
        print(f"    ← The cross-reference to Article 20 is split off from its context.")

print()
print("Structure-aware split (respect sub-clause boundaries):")
print("  Chunk 1: Article header + sub-clause (a) + (b)")
print("  Chunk 2: sub-clauses (c) + (d) + (e)")
print("  Chunk 3: sub-clauses (f) + (g)")
print()
print("WHY IT MATTERS for legal RAG:")
print("  Query: 'What happens after a high-risk AI is placed on market?'")
print("  Naive:  Might retrieve chunk that has 'placed on the market' mid-sentence")
print("          but the obligation about Article 20 is in the next chunk")
print("  Structure-aware: sub-clause (f) is always a complete unit → full obligation retrieved")

In [ ]:
# GRAPH ALGORITHMS — PageRank on the citation graph
# Articles that are referenced by many others are more important
# This gives us a way to RANK chunks beyond pure semantic similarity

import networkx as nx

# Build a directed graph from our RDF graph
DG = nx.DiGraph()

# Add edges from our RDF graph (references edges)
ref_pred = PREDS["references"]
for s, p, o in g:
    if p == ref_pred:
        src = str(s).split("/")[-1]
        tgt = str(o).split("/")[-1]
        DG.add_edge(src, tgt)

print(f"Citation graph: {DG.number_of_nodes()} nodes, {DG.number_of_edges()} edges")
print()

# Run PageRank
# In this context: PageRank tells us which articles are most "important"
# because they are referenced by many other articles
if DG.number_of_edges() > 0:
    pagerank = nx.pagerank(DG, alpha=0.85, max_iter=100)
    
    # Sort by PageRank score
    top_nodes = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:15]
    
    print("Top 15 most important nodes by PageRank:")
    print("(High score = referenced by many other articles)")
    print()
    for rank, (node, score) in enumerate(top_nodes, 1):
        name = node.replace("_", " ")
        bar = "█" * int(score * 5000)
        print(f"  {rank:2}. {name:<30} {score:.5f} {bar}")
    
    print()
    print("HOW TO USE PageRank in RAG:")
    print("  When scoring retrieved chunks, add a PageRank bonus:")
    print("  final_score = semantic_score * 0.7 + pagerank_score * 0.3")
    print("  This ensures high-importance articles bubble up in results.")
    print("  Annex III and Article 5 should always rank highly for legal queries.")
else:
    print("Few edges in graph — need richer cross-reference metadata.")
    print("PageRank would be applied once we have complete reference extraction.")
    print()
    print("Concept: Articles referenced by many others (e.g., Annex III, Article 5)")
    print("         get higher PageRank → their chunks score higher in retrieval.")

In [ ]:
# Community Detection — cluster articles into logical topic groups
# This helps with query routing: "which cluster does this query belong to?"

if DG.number_of_edges() > 5:
    # Convert to undirected for community detection
    UG = DG.to_undirected()
    
    # Louvain community detection (best for legal citation graphs)
    try:
        import community as community_louvain
        partition = community_louvain.best_partition(UG)
        communities = {}
        for node, comm_id in partition.items():
            communities.setdefault(comm_id, []).append(node)
        print(f"Found {len(communities)} communities via Louvain")
    except ImportError:
        # Fallback: connected components
        components = list(nx.connected_components(UG))
        communities = {i: list(c) for i, c in enumerate(components)}
        print(f"Found {len(communities)} connected components (install python-louvain for better clustering)")
    
    for comm_id, members in list(communities.items())[:5]:
        names = [m.replace("_", " ") for m in members[:5]]
        print(f"  Community {comm_id}: {', '.join(names)}{'...' if len(members) > 5 else ''}")
else:
    print("Community Detection explanation:")
    print()
    print("Louvain algorithm groups articles into clusters based on citation density.")
    print("Expected clusters for EU AI Act:")
    print()
    clusters = [
        ("Cluster A — High-Risk AI Definition",  ["Article 5", "Article 6", "Annex III", "Annex I", "Article 7"]),
        ("Cluster B — Provider Obligations",      ["Article 16", "Article 17", "Article 9", "Article 11", "Article 13"]),
        ("Cluster C — Conformity Assessment",     ["Article 43", "Article 44", "Article 40", "Article 41", "Annex VII"]),
        ("Cluster D — Governance",               ["Article 70", "Article 71", "Article 72", "Article 73", "Article 74"]),
        ("Cluster E — General Purpose AI",       ["Article 51", "Article 52", "Article 53", "Article 54", "Article 55"]),
    ]
    for name, members in clusters:
        print(f"  {name}")
        print(f"    Members: {', '.join(members)}")
        print()
    print("Use: Query 'What are provider obligations?' → routes to Cluster B → only that cluster is searched first.")

In [ ]:
# LOSSLESS EVALUATION FRAMEWORK
# The core problem: how do you know you haven't missed anything?

# Step 1: Obligation/Prohibition/Permission (OPP) triple extraction
# Every legal text can be decomposed into deontic logic triples:
#   OBLIGATION: "Providers SHALL..."
#   PROHIBITION: "AI systems MUST NOT..."
#   PERMISSION:  "Providers MAY..."

import re

def extract_deontic_triples(text: str, source: str) -> list:
    """
    Extract obligation/prohibition/permission triples from legal text.
    Returns list of {type, subject, obligation, source}
    """
    triples = []
    sentences = re.split(r'(?<=[.;])\s+', text)
    
    OBLIGATION_PATTERNS = [
        (r'shall\s+(.+?)(?=\.|;|$)', 'OBLIGATION'),
        (r'must\s+(.+?)(?=\.|;|$)',  'OBLIGATION'),
        (r'are required to\s+(.+?)(?=\.|;|$)', 'OBLIGATION'),
    ]
    PROHIBITION_PATTERNS = [
        (r'shall not\s+(.+?)(?=\.|;|$)',  'PROHIBITION'),
        (r'must not\s+(.+?)(?=\.|;|$)',   'PROHIBITION'),
        (r'prohibited\s+(.+?)(?=\.|;|$)', 'PROHIBITION'),
        (r'are prohibited from\s+(.+?)(?=\.|;|$)', 'PROHIBITION'),
    ]
    PERMISSION_PATTERNS = [
        (r'may\s+(.+?)(?=\.|;|$)',    'PERMISSION'),
        (r'can\s+(.+?)(?=\.|;|$)',    'PERMISSION'),
        (r'are allowed to\s+(.+?)(?=\.|;|$)', 'PERMISSION'),
    ]
    
    all_patterns = OBLIGATION_PATTERNS + PROHIBITION_PATTERNS + PERMISSION_PATTERNS
    
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) < 20:
            continue
        
        for pattern, dtype in all_patterns:
            match = re.search(pattern, sentence, re.IGNORECASE)
            if match:
                # Extract subject (words before the modal verb)
                modal_pos = match.start()
                subject_text = sentence[:modal_pos].strip()
                subject = subject_text[-60:] if len(subject_text) > 60 else subject_text
                obligation = match.group(1)[:100].strip()
                
                triples.append({
                    "type":       dtype,
                    "subject":    subject,
                    "obligation": obligation,
                    "source":     source,
                    "sentence":   sentence[:200]
                })
    
    return triples

# Test on Article 16 text
art16_text = """
Providers of high-risk AI systems shall ensure that their high-risk AI systems are compliant 
with the requirements set out in Section 2. Providers shall have a quality management system 
in place pursuant to Article 17. Providers shall not place on the market or put into service 
a high-risk AI system that does not meet these requirements. Providers may request information 
from notified bodies regarding conformity assessments. The use of real-time biometric identification 
systems in publicly accessible spaces for law enforcement purposes is prohibited.
"""

triples = extract_deontic_triples(art16_text, "Article 16")

print("Deontic Triple Extraction on Article 16:")
print("-" * 55)
for t in triples:
    icon = {"OBLIGATION": "🔵", "PROHIBITION": "🔴", "PERMISSION": "🟢"}.get(t['type'], "⚪")
    print(f"  {icon} {t['type']}")
    print(f"     Subject:    {t['subject'][:60]}")
    print(f"     Obligation: {t['obligation'][:80]}")
    print()

In [ ]:
# COVERAGE SCORE — the core of lossless evaluation
# After retrieving chunks for a query, measure:
# "What fraction of relevant deontic triples did we retrieve?"

def compute_coverage_score(
    retrieved_chunks: list,
    all_chunks: list,
    query_keywords: list
) -> dict:
    """
    Coverage score: fraction of relevant triples captured in retrieved chunks.
    
    1. Find ALL chunks that contain the query keywords → 'relevant universe'
    2. Extract deontic triples from ALL relevant chunks
    3. Extract deontic triples from RETRIEVED chunks
    4. Coverage = |retrieved triples| / |total relevant triples|
    """
    # Step 1: Find the 'relevant universe' by keyword match
    relevant_chunks = []
    for chunk in all_chunks:
        text = chunk.get("page_content", "").lower()
        if all(kw.lower() in text for kw in query_keywords):
            relevant_chunks.append(chunk)
    
    if not relevant_chunks:
        return {"coverage": 0.0, "total_relevant": 0, "retrieved_relevant": 0}
    
    # Step 2: Extract all deontic triples from relevant universe
    universe_triples = []
    for chunk in relevant_chunks:
        source = chunk["metadata"].get("source", "unknown")
        ts = extract_deontic_triples(chunk["page_content"], source)
        universe_triples.extend(ts)
    
    # Step 3: Extract triples from what we actually retrieved
    retrieved_triples = []
    retrieved_sources = set()
    for chunk in retrieved_chunks:
        source = chunk["metadata"].get("source", "unknown")
        retrieved_sources.add(source)
        ts = extract_deontic_triples(chunk["page_content"], source)
        retrieved_triples.extend(ts)
    
    # Step 4: Coverage
    universe_set = set(t['obligation'][:50] for t in universe_triples)
    retrieved_set = set(t['obligation'][:50] for t in retrieved_triples)
    
    covered = universe_set & retrieved_set
    missed  = universe_set - retrieved_set
    
    coverage = len(covered) / len(universe_set) if universe_set else 1.0
    
    return {
        "coverage_score":     round(coverage, 3),
        "total_relevant_chunks": len(relevant_chunks),
        "retrieved_count":    len(retrieved_chunks),
        "total_triples_in_universe": len(universe_set),
        "triples_covered":    len(covered),
        "triples_missed":     len(missed),
        "missed_obligations": list(missed)[:5],  # show first 5 missed
    }

# Test coverage for a query about provider obligations
# Let's use our actual chunks
retrieved_sample = [c for c in chunks if 
                    'provider' in c['page_content'].lower() and
                    c['metadata'].get('content_type') == 'article'][:5]

coverage_result = compute_coverage_score(
    retrieved_chunks=retrieved_sample,
    all_chunks=chunks,
    query_keywords=["provider", "shall"]
)

print("Coverage Score for query 'provider obligations':")
print("-" * 50)
for k, v in coverage_result.items():
    print(f"  {k:<35}: {v}")
print()
print("IF coverage < 0.8 → expand retrieval (increase k, add graph expansion)")
print("IF coverage >= 0.9 → we've captured enough for a lossless answer")
print()
print("This is how you answer: 'How do you ensure you don't miss any points?'")
print("The coverage score tells you if your retrieval is complete.")

In [ ]:
# CITATION TRACING — zero hallucination guarantee
# Every claim in the LLM answer must be traceable to a specific chunk

import re

def verify_citations(answer: str, retrieved_chunks: list) -> dict:
    """
    Verify that every [SOURCE N] citation in the answer actually appears
    in the retrieved chunks AND that the claim is grounded in that source.
    """
    # Find all [SOURCE N] citations in the answer
    cited_sources = re.findall(r'\[SOURCE (\d+)\]', answer)
    cited_indices = set(int(n) - 1 for n in cited_sources)  # 0-indexed
    
    results = {
        "total_citations":   len(cited_sources),
        "unique_sources":    len(set(cited_sources)),
        "valid_citations":   0,
        "invalid_citations": [],
        "uncited_chunks":    [],
        "coverage":          0.0,
    }
    
    for idx in cited_indices:
        if idx < len(retrieved_chunks):
            results["valid_citations"] += 1
        else:
            results["invalid_citations"].append(idx + 1)
    
    # Which chunks were never cited? (potential missing information)
    all_indices = set(range(len(retrieved_chunks)))
    uncited = all_indices - cited_indices
    for idx in sorted(uncited):
        source = retrieved_chunks[idx]["metadata"].get("source", f"Chunk {idx+1}")
        results["uncited_chunks"].append(source)
    
    if cited_indices:
        results["coverage"] = results["valid_citations"] / len(cited_indices)
    
    return results

# Simulate an LLM answer with citations
simulated_answer = """
Providers of high-risk AI systems have extensive obligations under the EU AI Act.

**Compliance**: Providers shall ensure their systems meet all requirements in Chapter III, Section 2 [SOURCE 1].

**Quality Management**: A quality management system must be established under Article 17 [SOURCE 2].

**Technical Documentation**: Providers must prepare documentation per Article 11 and Annex IV [SOURCE 1].

**Registration**: The AI system must be registered in the EU database [SOURCE 3].

**Post-market monitoring**: Providers shall take corrective actions if needed and report to authorities [SOURCE 5].
"""

# Simulated retrieved chunks (normally from ChromaDB)
simulated_chunks = [
    {"metadata": {"source": "Article 16 [1/3] | CHAPTER III, Page 62"}},
    {"metadata": {"source": "Article 17 [1/2] | CHAPTER III, Page 62"}},
    {"metadata": {"source": "Article 71 — EU Database | CHAPTER VIII, Page 88"}},
    {"metadata": {"source": "Recital (57) — providers shall put in place..."}},
    # Note: SOURCE 5 would be out of range (only 4 chunks)
]

citation_check = verify_citations(simulated_answer, simulated_chunks)

print("Citation Verification:")
print("-" * 45)
for k, v in citation_check.items():
    print(f"  {k:<25}: {v}")
print()
if citation_check["invalid_citations"]:
    print(f"  ⚠️  Invalid citations detected: SOURCE {citation_check['invalid_citations']}")
    print(f"     These citations don't exist in the retrieved context!")
    print(f"     This is a HALLUCINATION — the LLM fabricated a source.")
if citation_check["uncited_chunks"]:
    print(f"  ℹ️  Uncited chunks (may contain relevant info):")
    for s in citation_check["uncited_chunks"]:
        print(f"     - {s}")

In [ ]:
# CONTRADICTION DETECTION
# The LLM answer must not contradict any retrieved chunk
# This is done by NLI (Natural Language Inference)

# The model: cross-encoder/nli-deberta-v3-small
# Labels: ENTAILMENT, NEUTRAL, CONTRADICTION

print("Contradiction Detection Architecture")
print("=" * 50)
print()
print("Tool: NLI (Natural Language Inference) model")
print("  Model: cross-encoder/nli-deberta-v3-small (fast)")
print("  OR:    facebook/bart-large-mnli (more accurate)")
print()
print("Process:")
print("  For each claim in the answer:")
print("    For each retrieved chunk:")
print("      NLI(premise=chunk, hypothesis=claim) → [ENTAIL, NEUTRAL, CONTRADICT]")
print("      If CONTRADICT → flag as hallucination")
print()

# Mock NLI to show the concept
def mock_nli(premise: str, hypothesis: str) -> dict:
    """Mock NLI — in production use cross-encoder/nli-deberta-v3-small."""
    # Very simplified keyword-based mock
    p = premise.lower()
    h = hypothesis.lower()
    
    # Check for direct contradiction signals
    contradictions = [
        ('shall not', 'shall'),
        ('prohibited', 'allowed'),
        ('must not', 'may'),
    ]
    for neg, pos in contradictions:
        if neg in p and pos in h and neg not in h:
            return {"label": "CONTRADICTION", "score": 0.87}
    
    # Check for entailment signals
    common_words = set(p.split()) & set(h.split()) - {'the', 'a', 'and', 'or', 'of', 'to', 'in'}
    if len(common_words) > 4:
        return {"label": "ENTAILMENT", "score": 0.75}
    
    return {"label": "NEUTRAL", "score": 0.60}

# Test cases
test_cases = [
    {
        "chunk": "Real-time remote biometric identification systems in publicly accessible spaces are prohibited for law enforcement.",
        "claim": "Law enforcement authorities may use real-time biometric systems in public spaces with prior authorisation.",
        "expected": "CONTRADICTION"
    },
    {
        "chunk": "Providers of high-risk AI systems shall establish a quality management system.",
        "claim": "Providers must have a quality management system under Article 17.",
        "expected": "ENTAILMENT"
    },
    {
        "chunk": "The regulation enters into force 20 days after publication.",
        "claim": "Providers must register their systems in the EU database.",
        "expected": "NEUTRAL"
    },
]

print("NLI Test Cases:")
print("-" * 55)
for i, tc in enumerate(test_cases, 1):
    result = mock_nli(tc['chunk'], tc['claim'])
    icon = "✅" if result['label'] == tc['expected'] else "❌"
    print(f"  Case {i}: {icon} Expected {tc['expected']}, Got {result['label']} ({result['score']:.2f})")
    print(f"  Chunk:  {tc['chunk'][:70]}")
    print(f"  Claim:  {tc['claim'][:70]}")
    if result['label'] == 'CONTRADICTION':
        print(f"  ⚠️  HALLUCINATION DETECTED — claim contradicts source!")
    print()

print()
print("In production: replace mock_nli() with real NLI model:")
print("  from transformers import pipeline")
print("  nli = pipeline('text-classification', model='cross-encoder/nli-deberta-v3-small')")
print("  result = nli({'text': chunk, 'text_pair': claim})")

In [ ]:
# FULL LOSSLESS EVALUATION PIPELINE
# Putting it all together — this is what you describe in the interview

def lossless_evaluation_pipeline(query: str, answer: str, 
                                  retrieved_chunks: list,
                                  all_chunks: list) -> dict:
    """
    Complete lossless evaluation:
    1. Coverage score  — did we retrieve all relevant chunks?
    2. Citation check  — are all citations valid?
    3. NLI check       — any contradictions between answer and sources?
    4. Deontic audit   — did we capture all obligations/prohibitions?
    """
    print(f"\nLossless Evaluation")
    print(f"Query: {query[:60]}")
    print("=" * 55)
    
    # 1. Coverage
    keywords = [w for w in query.lower().split() if len(w) > 4]
    coverage = compute_coverage_score(retrieved_chunks, all_chunks, keywords[:3])
    print(f"\n1. COVERAGE SCORE: {coverage['coverage_score']:.1%}")
    print(f"   ({coverage['triples_covered']}/{coverage['total_triples_in_universe']} deontic triples captured)")
    if coverage['triples_missed']:
        print(f"   Missed obligations: {coverage['missed_obligations'][:3]}")
    
    # 2. Citations
    citations = verify_citations(answer, retrieved_chunks)
    print(f"\n2. CITATION CHECK: {citations['valid_citations']}/{len(citations['unique_sources']) if citations['unique_sources'] else 0} valid")
    if citations['invalid_citations']:
        print(f"   ⚠️  Invalid: SOURCE {citations['invalid_citations']} (potential hallucination!)")
    
    # 3. Deontic extraction from answer vs sources
    answer_triples = extract_deontic_triples(answer, "LLM Answer")
    print(f"\n3. DEONTIC AUDIT:")
    print(f"   Obligations in answer:   {sum(1 for t in answer_triples if t['type']=='OBLIGATION')}")
    print(f"   Prohibitions in answer:  {sum(1 for t in answer_triples if t['type']=='PROHIBITION')}")
    print(f"   Permissions in answer:   {sum(1 for t in answer_triples if t['type']=='PERMISSION')}")
    
    # 4. Overall verdict
    problems = []
    if coverage['coverage_score'] < 0.7:
        problems.append(f"Low coverage ({coverage['coverage_score']:.0%}) — expand retrieval")
    if citations['invalid_citations']:
        problems.append(f"Invalid citations — potential hallucination")
    
    print(f"\n4. VERDICT:")
    if not problems:
        print(f"   ✅ LOSSLESS — answer appears complete and grounded")
    else:
        print(f"   ❌ ISSUES FOUND:")
        for p in problems:
            print(f"      - {p}")
    
    return {
        "coverage": coverage,
        "citations": citations,
        "deontic_triples": answer_triples,
        "problems": problems,
        "is_lossless": len(problems) == 0
    }

# Run it
eval_result = lossless_evaluation_pipeline(
    query="What obligations does a provider of high-risk AI have?",
    answer=simulated_answer,
    retrieved_chunks=simulated_chunks,
    all_chunks=chunks
)

---
# The Complete Architecture — Interview Answer

Here's the full architecture you'd describe in an interview for a complex legal document RAG system.

## Ingestion Layer

```
PDF documents
    │
    ├─── [fitz layout parser]
    │         ├── Text blocks → structure-aware chunking
    │         │       (respect article/paragraph/sub-clause boundaries)
    │         ├── Image regions → VLM (GPT-4o / Qwen2-VL) → captions
    │         └── Table regions → pdfplumber / TATR → markdown tables
    │
    ├─── [ColPali as alternative]
    │         └── Render each page → PaliGemma embed → page-level index
    │
    └─── [Knowledge Graph builder]
              ├── Extract Article/Annex/Recital cross-references
              ├── Build RDF graph (rdflib)
              └── Add external documents (GDPR, Machinery Directive)
```

## Embedding Layer

```
Text chunks   → BGE-M3 (dense + sparse + colbert) → ChromaDB
Image chunks  → CLIP vision encoder               → ChromaDB (separate collection)
Table chunks  → BGE-M3 on markdown repr           → ChromaDB
Page images   → PaliGemma (ColPali)               → Vespa / Weaviate
```

## Retrieval Layer

```
Query
 │
 ├── [Multi-query expansion] → 3 sub-queries
 ├── [Legal vocabulary injection] → formal terms
 ├── [Query routing] → which cluster?
 │
 ├── [Dense search]  BGE-M3 → top-k text chunks
 ├── [Sparse search] BM25 → top-k lexical matches
 ├── [Image search]  CLIP → top-k image/table chunks
 │
 ├── [RRF fusion] → merged ranked list
 ├── [Graph expansion] SPARQL → 1-hop neighbors added
 ├── [PageRank boost] → importance-weighted reranking
 └── [Cross-encoder reranker] → final top-k
```

## Generation + Evaluation Layer

```
Context assembly (text + table text repr + image captions)
 │
 └── LLM generation with [SOURCE N] citations
      │
      ├── [Coverage Score]        — did we retrieve all relevant triples?
      ├── [Citation Verification] — all [SOURCE N] point to real chunks?
      ├── [NLI Contradiction]     — any claim contradicts source?
      └── [Deontic Audit]         — obligations/prohibitions extracted correctly?
```

## Key Terminology for the Interview

| Term | What it means |
|---|---|
| **Structure-aware chunking** | Split at logical boundaries (article → paragraph → sub-clause), never mid-obligation |
| **Graph RAG** | Use knowledge graph + SPARQL for cross-reference expansion during retrieval |
| **SPARQL** | SQL for knowledge graphs — query article→annex→recital relationships |
| **RDF** | Data format for knowledge graphs — (subject, predicate, object) triples |
| **PageRank** | Graph algorithm scoring article importance by citation count |
| **Community detection** | Louvain algorithm clusters articles into topic groups for query routing |
| **ColPali** | Embed PDF pages as images directly — no parsing, no chunking |
| **VLM** | Vision Language Model (GPT-4o, Qwen2-VL, LLaVA) — reads tables/images |
| **BGE-M3** | Best open-source legal embedding — dense + sparse + colbert in one model |
| **NLI** | Natural Language Inference — detects contradictions between answer and sources |
| **Deontic logic** | Obligation (SHALL) / Prohibition (SHALL NOT) / Permission (MAY) extraction |
| **Coverage score** | Fraction of relevant deontic triples captured in retrieved chunks |
| **Lossless** | Zero missed obligations, zero hallucinations, every claim cited |
| **RRF** | Reciprocal Rank Fusion — merge multiple ranked lists without score normalization |
| **TATR** | Table Transformer — ML model for detecting table regions in document images |

In [ ]:
print("WHAT TO BUILD NEXT")
print("=" * 55)
print()
print("Based on this deep dive, here's the build order:")
print()

next_steps = [
    ("1", "eu_ai_act_multimodal_extraction.ipynb",
     "Extract tables + images with exact bboxes, build MultimodalChunk store"),
    ("2", "eu_ai_act_colpali.ipynb",
     "Render all pages as images, embed with CLIP/PaliGemma, page-level retrieval"),
    ("3", "eu_ai_act_knowledge_graph.ipynb",
     "Build full RDF graph, SPARQL queries, 2-hop expansion, PageRank scoring"),
    ("4", "eu_ai_act_lossless_eval.ipynb",
     "Deontic extraction, NLI contradiction detection, coverage scoring framework"),
    ("5", "eu_ai_act_graph_rag.ipynb",
     "Full pipeline: hybrid search + SPARQL expansion + PageRank + lossless eval"),
]

for num, name, desc in next_steps:
    print(f"  Step {num}: {name}")
    print(f"           {desc}")
    print()

print("Each notebook builds on the previous.")
print("By step 5, you have a system that can answer any EU AI Act question")
print("with provenance (page + bbox), zero hallucination, and coverage guarantee.")
print()
print("That's a senior-level RAG system — and now you know every piece of it.")